<a href="https://colab.research.google.com/github/KoeJuntz/ResearchProject/blob/main/01_api_feasibility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kalshi API 10/16/2026

In [1]:
import requests
import pandas as pd

BASE = "https://external-api.kalshi.com/trade-api/v2"

def get_data(path, params=None):
    response = requests.get(
        BASE + path,
        params=params,
        timeout=30
    )
    response.raise_for_status()
    return response.json()

print(get_data("/historical/cutoff"))


{'market_positions_last_updated_ts': '2026-08-17T00:00:00Z', 'market_settled_ts': '2026-08-17T00:00:00Z', 'orders_updated_ts': '2026-10-02T00:00:00Z', 'trades_created_ts': '2026-08-17T00:00:00Z'}


In [2]:
markets = []
cursor = None

for page in range(20):
    params = {
        "series_ticker": "KXFEDDECISION",
        "limit": 100
    }

    if cursor:
        params["cursor"] = cursor

    result = get_data("/historical/markets", params)
    markets.extend(result["markets"])
    cursor = result.get("cursor")

    if not cursor:
        break

assert not cursor, "Inventory incomplete: pagination limit reached."

inventory = (
    pd.DataFrame(markets)
    .drop_duplicates("ticker")
)

print("Contracts:", len(inventory))
print("Event identifiers:", inventory["event_ticker"].nunique())

display(
    inventory.groupby("event_ticker")
    .agg(
        contracts=("ticker", "count"),
        close_time=("close_time", "min")
    )
    .sort_values("close_time")
)

Contracts: 140
Event identifiers: 28


,contracts,close_time
event_ticker,,
FEDDECISION-23MAY,6,2023-05-03T17:55:00Z
FEDDECISION-23JUN,5,2023-06-14T17:55:00Z
FEDDECISION-23JUL,5,2023-07-26T17:55:00Z
FEDDECISION-23SEP,5,2023-09-20T17:55:00Z
FEDDECISION-23NOV,5,2023-11-01T17:55:00Z
FEDDECISION-23DEC,5,2023-12-13T18:55:00Z
FEDDECISION-24JAN,5,2024-01-26T18:55:00Z
FEDDECISION-24JAN31,5,2024-01-31T18:55:00Z
FEDDECISION-24MAR20,5,2024-03-20T17:55:00Z


In [3]:
event_cutoffs = {
    "FEDDECISION-23MAY": "2023-05-02T16:00:00-04:00",
    "KXFEDDECISION-24DEC": "2024-12-17T16:00:00-05:00",
    "KXFEDDECISION-26JUL": "2026-07-28T16:00:00-04:00",
}

rows = []

for event, cutoff_text in event_cutoffs.items():
    cutoff = pd.Timestamp(cutoff_text).tz_convert("UTC")

    contracts = inventory.loc[
        inventory["event_ticker"].eq(event)
    ]

    assert not contracts.empty, f"Event missing: {event}"

    for _, contract in contracts.iterrows():
        ticker = contract["ticker"]

        result = get_data(
            f"/historical/markets/{ticker}/candlesticks",
            {
                "start_ts": int(
                    (cutoff - pd.Timedelta(days=7)).timestamp()
                ),
                "end_ts": int(cutoff.timestamp()),
                "period_interval": 60,
            }
        )

        candles = result.get("candlesticks", [])

        valid = [
            candle for candle in candles
            if candle["end_period_ts"] <= cutoff.timestamp()
            and candle.get("price", {}).get("close") is not None
        ]

        row = {
            "event": event,
            "contract": ticker,
            "outcome": contract.get("yes_sub_title"),
            "cutoff_utc": cutoff,
            "candles_retrieved": len(candles),
        }

        if valid:
            latest = max(valid, key=lambda c: c["end_period_ts"])

            timestamp = pd.to_datetime(
                latest["end_period_ts"], unit="s", utc=True
            )

            row.update({
                "candle_end_utc": timestamp,
                "yes_price_dollars": float(latest["price"]["close"]),
                "candle_volume": float(latest["volume"]),
                "candle_age_hours": (
                    cutoff - timestamp
                ).total_seconds() / 3600,
                "status": "retrieved",
            })
        else:
            row["status"] = "no priced candle in lookback"

        rows.append(row)

observations = pd.DataFrame(rows)

display(observations)

priced = observations.loc[
    observations["status"].eq("retrieved")
]

assert priced["yes_price_dollars"].between(0, 1).all()
assert priced["candle_age_hours"].ge(0).all()

print(
    "Events with retrieved observations:",
    priced["event"].nunique()
)

,event,contract,outcome,cutoff_utc,candles_retrieved,status,candle_end_utc,yes_price_dollars,candle_volume,candle_age_hours
0,FEDDECISION-23MAY,FEDDECISION-23MAY-TH50,Hike >50bps,2023-05-02 20:00:00+00:00,0,no priced candle in lookback,NaT,NaN,NaN,NaN
1,FEDDECISION-23MAY,FEDDECISION-23MAY-H50,Hike 50bps,2023-05-02 20:00:00+00:00,1,retrieved,2023-05-02 16:00:00+00:00,0.01,467.00,4.0
2,FEDDECISION-23MAY,FEDDECISION-23MAY-H25,Hike 25bps,2023-05-02 20:00:00+00:00,70,retrieved,2023-05-02 20:00:00+00:00,0.93,236.00,0.0
3,FEDDECISION-23MAY,FEDDECISION-23MAY-H0,No change,2023-05-02 20:00:00+00:00,59,retrieved,2023-05-02 20:00:00+00:00,0.06,207.00,0.0
4,FEDDECISION-23MAY,FEDDECISION-23MAY-TC25,Cut >25bps,2023-05-02 20:00:00+00:00,8,retrieved,2023-04-28 18:00:00+00:00,0.01,222.00,98.0
5,FEDDECISION-23MAY,FEDDECISION-23MAY-C25,Cut 25bps,2023-05-02 20:00:00+00:00,0,no priced candle in lookback,NaT,NaN,NaN,NaN
6,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H26,Hike >25bps,2024-12-17 21:00:00+00:00,2,retrieved,2024-12-12 07:00:00+00:00,0.01,4674.00,134.0
7,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H25,Hike 25bps,2024-12-17 21:00:00+00:00,9,retrieved,2024-12-17 08:00:00+00:00,0.01,100.00,13.0
8,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-H0,No cut/hike,2024-12-17 21:00:00+00:00,104,retrieved,2024-12-17 21:00:00+00:00,0.03,7405.00,0.0
9,KXFEDDECISION-24DEC,KXFEDDECISION-24DEC-C25,Cut 25bps,2024-12-17 21:00:00+00:00,123,retrieved,2024-12-17 21:00:00+00:00,0.97,5464.00,0.0


Events with retrieved observations: 3
